In [1]:
import pandas as pd

# Caminho da base do IBGE (2010 a 2021)
caminho_pib_recente = r'C:\Users\fabio\TCC\PIB dos Municípios - base de dados 2010-2021.xlsx'

try:
    # Lendo só as 5 primeiras linhas pra descobrir o nome exato das colunas.
    # (Ler o excel inteiro demora demais e pode travar a memória)
    df_pib_amostra = pd.read_excel(caminho_pib_recente, nrows=5)

    # Imprime a lista de colunas pra eu copiar e colar depois
    print("--- Colunas encontradas no arquivo 2010-2021 ---")
    print(df_pib_amostra.columns.tolist())

except Exception as e:
    print(f"Deu erro ao ler o arquivo do PIB: {e}")

--- Colunas encontradas no arquivo 2010-2021 ---
['Ano', 'Código da Grande Região', 'Nome da Grande Região', 'Código da Unidade da Federação', 'Sigla da Unidade da Federação', 'Nome da Unidade da Federação', 'Código do Município', 'Nome do Município', 'Região Metropolitana', 'Código da Mesorregião', 'Nome da Mesorregião', 'Código da Microrregião', 'Nome da Microrregião', 'Código da Região Geográfica Imediata', 'Nome da Região Geográfica Imediata', 'Município da Região Geográfica Imediata', 'Código da Região Geográfica Intermediária', 'Nome da Região Geográfica Intermediária', 'Município da Região Geográfica Intermediária', 'Código Concentração Urbana', 'Nome Concentração Urbana', 'Tipo Concentração Urbana', 'Código Arranjo Populacional', 'Nome Arranjo Populacional', 'Hierarquia Urbana', 'Hierarquia Urbana (principais categorias)', 'Código da Região Rural', 'Nome da Região Rural', 'Região rural (segundo classificação do núcleo)', 'Amazônia Legal', 'Semiárido', 'Cidade-Região de São Paul

In [2]:
# --- LIÇÃO 2: CARREGANDO E JUNTANDO OS DOIS ARQUIVOS DE PIB ---

import pandas as pd

# Caminhos dos arquivos do IBGE (separados por período)
caminho_recente = r'C:\Users\fabio\TCC\PIB dos Municípios - base de dados 2010-2021.xlsx'
caminho_antigo = r'C:\Users\fabio\TCC\PIB dos Municípios - base de dados 2002-2009.xlsx'

# Pegando só as colunas que importam pro modelo pra não estourar a RAM do PC.
# Obs: O IBGE coloca umas quebras de linha (\n) bizarras no nome, tem que copiar exatamente igual ao print anterior:
colunas_para_usar = [
    'Ano',
    'Código do Município',
    'Produto Interno Bruto, \na preços correntes\n(R$ 1.000)',
    'Produto Interno Bruto per capita, \na preços correntes\n(R$ 1,00)'
]

print("Carregando arquivo 2010-2021 (vai demorar um pouco porque é excel)...")
df_recente = pd.read_excel(caminho_recente, usecols=colunas_para_usar)

print("Carregando arquivo 2002-2009...")
# Assumindo que a base antiga manteve exatamente os mesmos nomes de coluna
df_antigo = pd.read_excel(caminho_antigo, usecols=colunas_para_usar)

# Empilhando as duas bases. 
# Lembrar do ignore_index=True pro pandas recalcular o índice de 0 até o final, senão vai ter índice duplicado.
df_pib_completo = pd.concat([df_antigo, df_recente], ignore_index=True)

# Conferindo se o empilhamento deu certo (tem que ir de 2002 a 2021)
print("\n--- Tabela de PIB Completa (2002-2021) ---")
print(df_pib_completo.head())
print(df_pib_completo.tail())

# Checando se o IBGE não mandou os valores do PIB como texto (string) por acidente
print("\n--- Tipos de dados ---")
df_pib_completo.info()

--- Carregando arquivo 2010-2021 (pode demorar um pouco)... ---
--- Carregando arquivo 2002-2009... ---

--- Tabela de PIB Completa (2002-2021) ---
Primeiras linhas (deve começar em 2002):
    Ano  Código do Município  \
0  2002              1100015   
1  2002              1100023   
2  2002              1100031   
3  2002              1100049   
4  2002              1100056   

   Produto Interno Bruto, \na preços correntes\n(R$ 1.000)  \
0                                         111290.995         
1                                         449592.816         
2                                          31767.520         
3                                         474443.097         
4                                          79173.614         

   Produto Interno Bruto per capita, \na preços correntes\n(R$ 1,00)  
0                                            4047.83                  
1                                            5667.37                  
2                               

C:\Users\fabio\anaconda3\Lib\site-packages\openpyxl\worksheet\header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


In [4]:
# --- LIÇÃO 3: PADRONIZANDO E SALVANDO O PIB REAL ---

# (df_pib_completo já tá na memória depois de rodar o script anterior)

# Dicionário pra limpar esses nomes bizarros cheios de quebra de linha (\n) do IBGE
# Já aproveitei pra chamar o código de 'id_municipio_6' pra bater certinho com as outras bases do TCC
dicionario_renomear = {
    'Código do Município': 'id_municipio_6',
    'Produto Interno Bruto, \na preços correntes\n(R$ 1.000)': 'PIB_Real_Mil_Reais',
    'Produto Interno Bruto per capita, \na preços correntes\n(R$ 1,00)': 'PIB_per_capita_Real'
}

# Aplica os nomes novos
df_pib_padrao = df_pib_completo.rename(columns=dicionario_renomear)

# O grande truque do IBGE: eles mandam o código com 7 dígitos, mas o resto das minhas bases (RAIS, DATASUS) usa 6.
# Transformando em string e cortando o último dígito pra não quebrar o Master Merge depois.
df_pib_padrao['id_municipio_6'] = df_pib_padrao['id_municipio_6'].astype(str).str[:6]

# Ano vira texto também pra evitar bug no cruzamento
df_pib_padrao['Ano'] = df_pib_padrao['Ano'].astype(str)

# Conferindo se o corte dos dígitos deu certo
print("--- Tabela de PIB Padronizada (Chaves como Texto) ---")
print(df_pib_padrao.head())
print("\n--- Verificando os tipos finais ---")
df_pib_padrao.info()

# Salvando direto na pasta de finalizados pra não misturar com os arquivos brutos
caminho_salvar = r'C:\Users\fabio\TCC\FINALIZADOS\12_PIB_FINAL.csv'
df_pib_padrao.to_csv(caminho_salvar, index=False) # index=False é lei pra não exportar aquela coluna inútil de índice

print(f"\n--- SUCESSO! ---")
print(f"Arquivo '12_PIB_FINAL.csv' salvo na pasta FINALIZADOS.")

--- Tabela de PIB Padronizada (Chaves como Texto) ---
    Ano Cod_IBGE  PIB_Real_Mil_Reais  PIB_per_capita_Real
0  2002   110001          111290.995              4047.83
1  2002   110002          449592.816              5667.37
2  2002   110003           31767.520              4246.99
3  2002   110004          474443.097              6353.27
4  2002   110005           79173.614              4442.47

--- Verificando os tipos finais ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 111326 entries, 0 to 111325
Data columns (total 4 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   Ano                  111326 non-null  object 
 1   Cod_IBGE             111326 non-null  object 
 2   PIB_Real_Mil_Reais   111326 non-null  float64
 3   PIB_per_capita_Real  111326 non-null  float64
dtypes: float64(2), object(2)
memory usage: 3.4+ MB

--- SUCESSO! ---
Arquivo 'pib_real_FINAL.csv' salvo na pasta FINALIZADOS.
